# BiLSTM Adversarial Examples for Train Set


In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, "../src")
# import from helper module to avoid redundancy in code
import generate_bilstm_adversarial as attack_impl


## Setup

The train notebook attacks a reproducible fraction of phishing URLs detected by the clean BiLSTM.

In [9]:
SPLIT = "train"
FRAC = 0.25
N = None  # set to a small number such as 10 for a smoke test
BATCH_SIZE = 64
TOP_POSITIONS = 3
SEED = 42

CHECKPOINT = Path("../models/bilstm_model1.pt")
OUTPUT = Path("../data/bilstm_adv_train.csv")
PARTIAL = Path("../data/bilstm_adv_train_partial.csv")
RESULTS_DIR = Path("../results")

config, vocab = attack_impl.load_config()
MAX_LEN = config["max_len"]
K = config["budget"]
INDEX_TO_STRING = vocab["index_to_string"]
CHAR_TO_IDX = {character: index for index, character in enumerate(INDEX_TO_STRING)}
PAD_IDX = vocab["pad_idx"]
UNK_IDX = vocab["unk_idx"]
torch = attack_impl.torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", DEVICE, "| split:", SPLIT, "| frac:", FRAC, "| k:", K)

device: cuda | split: train | frac: 0.25 | k: 3


## Load Data and Model

In [10]:
splits = attack_impl.load_splits()
model = attack_impl.load_model(CHECKPOINT, len(CHAR_TO_IDX), PAD_IDX, DEVICE)
embedding = model.embedding.weight
valid_embedding_ids = torch.tensor([index for character, index in CHAR_TO_IDX.items() if character not in {"<PAD>", "<UNK>"}], dtype=torch.long, device=DEVICE)
p_phish = attack_impl.make_predictor(model, CHAR_TO_IDX, UNK_IDX, PAD_IDX, MAX_LEN, DEVICE)

## Select a Reproducible Training Subset

In [11]:
phishing = attack_impl.drop_invalid_originals(splits[SPLIT][splits[SPLIT].label == 0]).copy()
p_all = p_phish(phishing.URL.tolist())
targets = phishing[p_all >= 0.5].copy()
targets["p_clean"] = p_all[p_all >= 0.5]
targets = targets.sample(frac=FRAC, random_state=SEED)
if N is not None:
    targets = targets.sample(min(N, len(targets)), random_state=SEED)
print(f"phishing: {len(phishing)} | missed: {(p_all < 0.5).sum()} | to attack: {len(targets)}")

phishing: 77552 | missed: 13954 | to attack: 15900


## Generate Character-Level Attacks

The model is frozen. Gradients are taken through the character embedding sequence, and candidate URLs are still validated against the original URL and the three-edit budget.

In [13]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
PARTIAL.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if PARTIAL.exists():
    completed = pd.read_csv(PARTIAL, keep_default_na=False)
else:
    completed = pd.DataFrame(columns=["orig_URL", "URL"])
todo = targets[~targets.URL.isin(completed.orig_URL)]
start_time = time.time()
for start in tqdm(range(0, len(todo), BATCH_SIZE), desc=f"attack {SPLIT}"):
    chunk = todo.iloc[start:start + BATCH_SIZE]
    adversarial = attack_impl.attack_batch(chunk.URL.tolist(), chunk.p_clean.tolist(), model, embedding, valid_embedding_ids, CHAR_TO_IDX, UNK_IDX, PAD_IDX, MAX_LEN, K, TOP_POSITIONS, DEVICE, p_phish)
    pd.DataFrame({"orig_URL": chunk.URL.values, "URL": adversarial}).to_csv(PARTIAL, mode="a", header=not PARTIAL.exists(), index=False)
print("seconds per URL:", round((time.time() - start_time) / max(1, len(todo)), 4))

attack train:   0%|          | 0/225 [00:00<?, ?it/s]

seconds per URL: 0.1227


## Validate, Summarize, and Save

In [15]:
attack_map = dict(pd.read_csv(PARTIAL, keep_default_na=False).values) if PARTIAL.exists() else {}
adv_df = pd.DataFrame({"orig_URL": targets.URL.values, "URL": [attack_map.get(url, url) for url in targets.URL], "label": 0, "p_clean": targets.p_clean.values})
adv_df["p_adv"] = p_phish(adv_df.URL.tolist())
adv_df["n_edits"] = [attack_impl.edit_distance(original, edited) for original, edited in zip(adv_df.orig_URL, adv_df.URL)]
adv_df["flipped"] = adv_df.p_adv < 0.5
reasons = pd.Series([attack_impl.invalid_reason(original, edited, K) for original, edited in zip(adv_df.orig_URL, adv_df.URL)])
print(reasons.replace("", "valid").value_counts())
invalid = reasons.ne("")
if invalid.any():
    print(f"Reverting {invalid.sum()} invalid attack result(s) to their original URLs.")
    adv_df.loc[invalid, "URL"] = adv_df.loc[invalid, "orig_URL"]

    adv_df["p_adv"] = p_phish(adv_df.URL.tolist())
    adv_df["n_edits"] = [
        attack_impl.edit_distance(original, edited)
        for original, edited in zip(adv_df.orig_URL, adv_df.URL)
    ]
    adv_df["flipped"] = adv_df.p_adv < 0.5
    reasons = pd.Series([
        attack_impl.invalid_reason(original, edited, K)
        for original, edited in zip(adv_df.orig_URL, adv_df.URL)
    ])

assert (reasons == "").all(), "Original URLs also failed validation"
adv_df["duplicate"] = (adv_df.n_edits > 0) & (adv_df.URL.isin(splits[SPLIT].URL) | adv_df.URL.duplicated())
summary = {"model": "bilstm", "split": SPLIT, "phishing": len(phishing), "missed_by_model": int((p_all < 0.5).sum()), **attack_impl.attack_summary(adv_df, K), "n_edits": {int(count): int(number) for count, number in adv_df.n_edits.value_counts().sort_index().items()}}
adv_df.to_csv(OUTPUT, index=False)
if PARTIAL.exists():
    PARTIAL.unlink()
with (RESULTS_DIR / "bilstm_adv_train_summary.json").open("w") as file:
    json.dump(summary, file, indent=1)
display(adv_df.head())
summary

valid            15899
bad character        1
Name: count, dtype: int64
Reverting 1 invalid attack result(s) to their original URLs.


AssertionError: Original URLs also failed validation

In [18]:
# splits[SPLIT] refers
# 
#
# to


original_urls = splits[SPLIT]["URL"].astype(str)
original_reasons = pd.Series(
    [attack_impl.invalid_reason(url, url, K) for url in original_urls],
    index=original_urls.index,
    name="invalid_reason",
)

invalid_originals = splits[SPLIT].loc[original_reasons.ne("")].copy()
invalid_originals["invalid_reason"] = original_reasons[original_reasons.ne("")]

print(f"Invalid original URLs: {len(invalid_originals)}")
display(invalid_originals)

normal_originals = splits[SPLIT].loc[original_reasons.eq("")].sample(
    n=min(10, original_reasons.eq("").sum()),
    random_state=SEED,
)
display(normal_originals)

Invalid original URLs: 2


,URL,label,invalid_reason
101551,22017-5502.s3.webspace.re/privatkunden/Ã£Å“ber...,0,bad character
125323,vinted.6545657.xyz/p5ci5kt9Ã¢â‚¬â€¹Ã¢â‚¬â€¹Ã¢â...,0,bad character


,URL,label
64825,3y.tmztp.com,0
11840,newsis.com,1
14460,coeh.berkeley.edu,1
92123,storage.cloud.google.com/rrdar99rt9qraraq99eus...,0
182047,fbfs.com,1
53510,spc.int,1
35355,giftsforknitters.com/media,0
149817,cosafa.com,1
35697,usujuegos.com,1
4124,9caef1c3.sibforms.com/serve/muieahhmyencciph6p...,0
